<a href="https://colab.research.google.com/github/Levan-Danelia/FRTB/blob/main/FRTB_CRCV_Non_Securitization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# --- Helper function for display ---
# In a real Jupyter environment, you would just type the DataFrame name.
# This function simulates that behavior for a script.
def display_df(df, title=""):
    """Prints a DataFrame with a title."""
    print(f"--- {title} ---")
    print(df.to_string())
    print("\n" + "="*50 + "\n")

# ==============================================================================
# Cell 1: Step 3 - Establish Gross Positions
# ==============================================================================
print("### Step 3: Establish Gross Positions ###\n")

# Define the initial portfolio of gross curvature risk positions
gross_positions_data = {
    'Bucket': [12, 12, 11, 11],
    'Issuer': ['Issuer A', 'Issuer A', 'Issuer B', 'Issuer C'],
    'Sector': ['Energy', 'Energy', 'Financials', 'Financials'],
    'CVR+': [2103, -10626, -30694, 25131],
    'CVR-': [276, -13025, 20905, 19853]
}
gross_positions_df = pd.DataFrame(gross_positions_data)

display_df(gross_positions_df, "Initial Gross Positions")

# ==============================================================================
# Cell 2: Step 4 - Calculate Net Positions
# ==============================================================================
print("### Step 4: Calculate Net Positions ###\n")
print("Netting positions for the same risk factor (Issuer) as per Article 325g(1).\n")

# Group by Bucket, Issuer, and Sector, then sum the CVR values
net_positions_df = gross_positions_df.groupby(['Bucket', 'Issuer', 'Sector']).sum().reset_index()

display_df(net_positions_df, "Final Net Positions")

# ==============================================================================
# Cell 3: Steps 5 & 6 - Determine Base Curvature Correlations
# ==============================================================================
print("### Steps 5 & 6: Determine Base Curvature Correlations (Medium Scenario) ###\n")
print("Curvature correlations are the square of the corresponding delta correlations (Article 325ay(5)).\n")

# --- Intra-Bucket Correlation (rho) ---
# Delta correlation between different names within the same CSR bucket is 35%
delta_rho_csr = 0.35
curvature_rho_medium = delta_rho_csr**2
print(f"Base Intra-Bucket Delta Correlation (rho_delta): {delta_rho_csr:.2%}")
print(f"Base Intra-Bucket Curvature Correlation (rho_curvature): {curvature_rho_medium:.4f} or {curvature_rho_medium:.2%}\n")


# --- Cross-Bucket Correlation (gamma) ---
# Delta correlation is 5% (rating factor 1 * sector factor 5%)
delta_gamma_csr = 0.05
curvature_gamma_medium = delta_gamma_csr**2
print(f"Base Cross-Bucket Delta Correlation (gamma_delta): {delta_gamma_csr:.2%}")
print(f"Base Cross-Bucket Curvature Correlation (gamma_curvature): {curvature_gamma_medium:.4f} or {curvature_gamma_medium:.2%}")
print("\n" + "="*50 + "\n")

# ==============================================================================
# Cell 4: Step 7 & 8 - Bucket Capital Calculation Logic
# ==============================================================================
print("### Steps 7 & 8: Bucket-Level Capital Calculation ###\n")

def psi(val1, val2):
    """Safeguard function from Article 325g(4)."""
    return 0 if val1 < 0 and val2 < 0 else 1

def calculate_bucket_capital(bucket_df, correlation):
    """
    Calculates K_b+, K_b-, the final K_b, and the selected scenario for a single bucket.
    Implements the logic from Article 325g(4).
    """
    # --- Upward Scenario (K_b+) ---
    sum_sq_cvr_plus = np.sum(np.maximum(bucket_df['CVR+'], 0)**2)

    correlation_term_plus = 0
    if len(bucket_df) > 1:
        # Simplified for 2 risk factors, as in our case
        cvr_k_plus = bucket_df['CVR+'].iloc[0]
        cvr_l_plus = bucket_df['CVR+'].iloc[1]
        correlation_term_plus = 2 * correlation * cvr_k_plus * cvr_l_plus * psi(cvr_k_plus, cvr_l_plus)

    kb_plus = np.sqrt(max(0, sum_sq_cvr_plus + correlation_term_plus))

    # --- Downward Scenario (K_b-) ---
    sum_sq_cvr_minus = np.sum(np.maximum(bucket_df['CVR-'], 0)**2)

    correlation_term_minus = 0
    if len(bucket_df) > 1:
        cvr_k_minus = bucket_df['CVR-'].iloc[0]
        cvr_l_minus = bucket_df['CVR-'].iloc[1]
        correlation_term_minus = 2 * correlation * cvr_k_minus * cvr_l_minus * psi(cvr_k_minus, cvr_l_minus)

    kb_minus = np.sqrt(max(0, sum_sq_cvr_minus + correlation_term_minus))

    # --- Final K_b and Scenario Selection ---
    kb_final = max(kb_plus, kb_minus)

    # Per Article 325g(4), if K_b+ == K_b-, selection is based on the sum of CVRs
    if kb_plus == kb_minus:
        selected_scenario = 'Upward' if bucket_df['CVR+'].sum() > bucket_df['CVR-'].sum() else 'Downward'
    else:
        selected_scenario = 'Upward' if kb_plus > kb_minus else 'Downward'

    return kb_plus, kb_minus, kb_final, selected_scenario

# Calculate for each bucket using medium correlation
bucket_results = {}
for bucket_id, group in net_positions_df.groupby('Bucket'):
    print(f"Calculating for Bucket {bucket_id}...")
    k_plus, k_minus, k_final, scenario = calculate_bucket_capital(group, curvature_rho_medium)
    bucket_results[bucket_id] = {
        'K_b+': k_plus,
        'K_b-': k_minus,
        'K_b': k_final,
        'Selected Scenario': scenario
    }
    print(f"  K_b+ = {k_plus:,.2f}")
    print(f"  K_b- = {k_minus:,.2f}")
    print(f"  Final K_b = {k_final:,.2f}")
    print(f"  Selected Scenario: {scenario}\n")

display_df(pd.DataFrame(bucket_results).T, "Summary of Bucket Capital (Medium Scenario)")

# ==============================================================================
# Cell 5: Step 9 - Determine Bucket Sums (S_b)
# ==============================================================================
print("### Step 9: Determine Bucket Sums (S_b) ###\n")
print("Calculating S_b based on the selected scenario for each bucket.\n")

bucket_sums = {}
for bucket_id, result in bucket_results.items():
    bucket_df = net_positions_df[net_positions_df['Bucket'] == bucket_id]
    if result['Selected Scenario'] == 'Upward':
        s_b = bucket_df['CVR+'].sum()
    else:
        s_b = bucket_df['CVR-'].sum()
    bucket_sums[bucket_id] = s_b

s_b_df = pd.DataFrame.from_dict(bucket_sums, orient='index', columns=['S_b'])
display_df(s_b_df, "Bucket Sums (S_b)")

# ==============================================================================
# Cell 6: Step 10 - Calculate Cross-Bucket Capital (Medium Scenario)
# ==============================================================================
print("### Step 10: Calculate Cross-Bucket Capital (Medium Scenario) ###\n")

# Sum of squared K_b values
sum_sq_kb = sum(res['K_b']**2 for res in bucket_results.values())

# Correlation term (simplified for 2 buckets)
s_11 = bucket_sums[11]
s_12 = bucket_sums[12]
cross_bucket_corr_term = 2 * curvature_gamma_medium * s_11 * s_12 * psi(s_11, s_12)

# Final RCCR for medium scenario
rccr_medium = np.sqrt(max(0, sum_sq_kb + cross_bucket_corr_term))

print(f"Sum of Squared K_b's = {sum_sq_kb:,.2f}")
print(f"Cross-Bucket Correlation Term = {cross_bucket_corr_term:,.2f}")
print(f"Final Capital (Medium Scenario) = sqrt({sum_sq_kb:,.0f} + {cross_bucket_corr_term:,.0f}) = {rccr_medium:,.2f}")
print("\n" + "="*50 + "\n")

# ==============================================================================
# Cell 7: Step 11 - Correlation Scenarios & Final Capital
# ==============================================================================
print("### Step 11: Correlation Scenarios & Final Capital ###\n")

def calculate_total_capital_for_scenario(scenario_name):
    """Calculates the total RCCR for a given scenario ('Low', 'Medium', 'High')."""

    # 1. Determine Scenario Correlations
    if scenario_name == 'High':
        rho_scen = curvature_rho_medium * 1.25
        gamma_scen = curvature_gamma_medium * 1.25
    elif scenario_name == 'Low':
        rho_scen = max(2 * curvature_rho_medium - 1, 0.75 * curvature_rho_medium)
        gamma_scen = max(2 * curvature_gamma_medium - 1, 0.75 * curvature_gamma_medium)
    else: # Medium
        rho_scen = curvature_rho_medium
        gamma_scen = curvature_gamma_medium

    # 2. Recalculate Bucket Capitals (K_b)
    scen_bucket_results = {}
    for bucket_id, group in net_positions_df.groupby('Bucket'):
        _, _, k_final, scenario = calculate_bucket_capital(group, rho_scen)
        scen_bucket_results[bucket_id] = {'K_b': k_final, 'Selected Scenario': scenario}

    # 3. Recalculate Bucket Sums (S_b)
    scen_bucket_sums = {}
    for bucket_id, result in scen_bucket_results.items():
        bucket_df = net_positions_df[net_positions_df['Bucket'] == bucket_id]
        scen_bucket_sums[bucket_id] = bucket_df['CVR+'].sum() if result['Selected Scenario'] == 'Upward' else bucket_df['CVR-'].sum()

    # 4. Recalculate Final RCCR
    sum_sq_kb_scen = sum(res['K_b']**2 for res in scen_bucket_results.values())
    s_11_scen = scen_bucket_sums[11]
    s_12_scen = scen_bucket_sums[12]
    cross_bucket_corr_term_scen = 2 * gamma_scen * s_11_scen * s_12_scen * psi(s_11_scen, s_12_scen)

    rccr_final = np.sqrt(max(0, sum_sq_kb_scen + cross_bucket_corr_term_scen))

    return {
        'Intra-Bucket Corr': rho_scen,
        'Cross-Bucket Corr': gamma_scen,
        'Total Capital (RCCR)': rccr_final
    }

# Calculate for all scenarios
scenario_results = {
    'Low': calculate_total_capital_for_scenario('Low'),
    'Medium': calculate_total_capital_for_scenario('Medium'),
    'High': calculate_total_capital_for_scenario('High')
}

scenario_df = pd.DataFrame(scenario_results).T
scenario_df['Intra-Bucket Corr'] = scenario_df['Intra-Bucket Corr'].map('{:.4%}'.format)
scenario_df['Cross-Bucket Corr'] = scenario_df['Cross-Bucket Corr'].map('{:.4%}'.format)

display_df(scenario_df, "Scenario Capital Results")

# Determine Final Capital Charge
final_capital_charge = scenario_df['Total Capital (RCCR)'].max()

print("### Final CSR Curvature Capital Requirement ###\n")
print(f"The final capital charge is the maximum of the three scenarios.\n")
print(f"Final Capital = max({scenario_df['Total Capital (RCCR)'].iloc[0]:,.2f}, {scenario_df['Total Capital (RCCR)'].iloc[1]:,.2f}, {scenario_df['Total Capital (RCCR)'].iloc[2]:,.2f})")
print(f"Final Capital Charge = {final_capital_charge:,.2f}")

### Step 3: Establish Gross Positions ###

--- Initial Gross Positions ---
   Bucket    Issuer      Sector   CVR+   CVR-
0      12  Issuer A      Energy   2103    276
1      12  Issuer A      Energy -10626 -13025
2      11  Issuer B  Financials -30694  20905
3      11  Issuer C  Financials  25131  19853


### Step 4: Calculate Net Positions ###

Netting positions for the same risk factor (Issuer) as per Article 325g(1).

--- Final Net Positions ---
   Bucket    Issuer      Sector   CVR+   CVR-
0      11  Issuer B  Financials -30694  20905
1      11  Issuer C  Financials  25131  19853
2      12  Issuer A      Energy  -8523 -12749


### Steps 5 & 6: Determine Base Curvature Correlations (Medium Scenario) ###

Curvature correlations are the square of the corresponding delta correlations (Article 325ay(5)).

Base Intra-Bucket Delta Correlation (rho_delta): 35.00%
Base Intra-Bucket Curvature Correlation (rho_curvature): 0.1225 or 12.25%

Base Cross-Bucket Delta Correlation (gamma_delta): 5.